# iREL NLP Task - Colab Notebook

**First time / fresh session:** Run all cells top to bottom.

**After pushing a code change:** Re-run cell 2 (pull) then cell 7 (run pipeline).

**Change video:** Edit `VIDEO_ID` in cell 1, then re-run cell 7.

In [ ]:
# 1) Settings — edit here if needed
REPO_URL    = "https://github.com/AnishRacherla/irel-nlp-task.git"
PROJECT_DIR = "/content/irel-nlp-task"
VIDEO_ID    = "video_2"   # video_1 / video_2 / video_3 / video_4
RUN_ALL     = False


In [ ]:
# 2) Clone repo (first time) or pull latest changes (subsequent runs)
import os, subprocess

if os.path.exists(f"{PROJECT_DIR}/.git"):
    print("Repo already exists — pulling latest changes...")
    result = subprocess.run(["git", "-C", PROJECT_DIR, "pull"], capture_output=True, text=True)
    print(result.stdout or "Already up to date.")
else:
    print("Cloning repo for the first time...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    print("Cloned.")

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())


In [ ]:
# 3) Write config.yaml with Groq API key
import os
os.chdir("/content/irel-nlp-task")
os.makedirs("config", exist_ok=True)

with open("config/config.yaml", "w") as f:
    f.write("""video_sources:
  video_1:
    url: \"https://www.youtube.com/watch?v=XV-lIaO00H8\"
    language: auto
    domain: Computer Science
    duration_minutes: 10
  video_2:
    url: \"https://www.youtube.com/watch?v=IlWB81vEH7g\"
    language: auto
    domain: Computer Science
    duration_minutes: 10
  video_3:
    url: \"https://www.youtube.com/watch?v=cOSTc6qBRQw\"
    language: auto
    domain: Computer Science
    duration_minutes: 10
  video_4:
    url: \"https://www.youtube.com/watch?v=SkE2kD2U4tU\"
    language: auto
    domain: Magnetism
    duration_minutes: 30

transcription:
  model: whisper
  whisper_model_size: base
  language: auto
  output_format: json

language_processing:
  detect_code_mixing: true
  primary_languages: [en, hi, ta, te, kn]
  standardization_method: hybrid

concept_extraction:
  method: hybrid
  llm_provider: groq
  model: gpt-4o-mini
  groq_model: llama-3.3-70b-versatile
  min_concept_confidence: 0.7
  max_concepts_per_video: 20

prerequisite_mapping:
  method: hybrid
  confidence_threshold: 0.6

output:
  format: json

api_keys:
  groq_api_key: \"YOUR_GROQ_API_KEY_HERE\"
  openai_api_key: \"\"
""")
print("config/config.yaml written.")


In [ ]:
# 4) Check GPU  (Runtime -> Change runtime type -> T4 GPU)
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 5) Install system + Python dependencies  <- run once per Colab session
import os
os.chdir("/content/irel-nlp-task")

!apt-get install -y ffmpeg -qq
!pip -q install -U pip
!pip -q install -r requirements.txt
!pip -q install groq


In [ ]:
# 6) Install local package + NLP models  <- run once per Colab session
import os
os.chdir("/content/irel-nlp-task")

!pip -q install -e .
!python -m spacy download en_core_web_sm

import nltk
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)

from src.pipeline import PedagogicalFlowPipeline
print("Environment ready.")


In [ ]:
# 7) Run pipeline  <- re-run this after any code change or to process a different video
import os, subprocess
os.chdir("/content/irel-nlp-task")

cmd = ["python", "main.py", "--process-all"] if RUN_ALL else \
      ["python", "example_usage.py", "--video-id", VIDEO_ID]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Pipeline failed — see output above.")


In [ ]:
# 8) Preview output JSON
import os, json
os.chdir("/content/irel-nlp-task")

json_path = f"outputs/graphs/{VIDEO_ID}_complete_output.json"
if not os.path.exists(json_path):
    candidates = [p for p in os.listdir("outputs/graphs") if p.endswith("_complete_output.json")]
    json_path = os.path.join("outputs/graphs", candidates[0])

with open(json_path) as f:
    data = json.load(f)

print("File:", json_path)
print("Concepts:", len(data.get("concepts", [])))
print("Relationships:", len(data.get("relationships", [])))
for c in data.get("concepts", [])[:5]:
    print(" -", c.get("name", c.get("id")))


In [ ]:
# 9) Show interactive graph
import os
from IPython.display import IFrame, display
os.chdir("/content/irel-nlp-task")

html_path = f"outputs/visualizations/{VIDEO_ID}_interactive_graph.html"
if not os.path.exists(html_path):
    candidates = [p for p in os.listdir("outputs/visualizations") if p.endswith("_interactive_graph.html")]
    html_path = os.path.join("outputs/visualizations", candidates[0])

display(IFrame(src=html_path, width=1100, height=650))


In [ ]:
# 10) Download all outputs as zip
import os, zipfile
from google.colab import files
os.chdir("/content/irel-nlp-task")

with zipfile.ZipFile("irel_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["outputs/graphs", "outputs/visualizations"]:
        for root, _, filenames in os.walk(folder):
            for name in filenames:
                path = os.path.join(root, name)
                zf.write(path)

files.download("irel_outputs.zip")


## Quick Reference

| Scenario | Cells to run |
|---|---|
| First time / new session | All cells (1 → 10) |
| You pushed a code change | Cell 2 (pull) → Cell 7 (run) |
| Change video ID | Edit `VIDEO_ID` in cell 1 → Cell 7 |
| Process all videos | Set `RUN_ALL = True` in cell 1 → Cell 7 |